# 🎯 Ćwiczenie 3: Spark SQL, DataFrames i zaawansowane transformacje

**Kurs:** Python w Zastosowaniach Big Data — PJATK  
**Laboratorium:** 3  
**Data:** 12.04.2026  
**Czas:** ⏱️ 90 minut  
**Poziom trudności:** ⭐⭐ (Intermediate)

## 📚 Cele dydaktyczne

Po ukończeniu tego notebooka będziesz potrafić:
1. Wykonywać zaawansowane operacje na **DataFrame** — filtrowanie, selekcja, transformacje kolumn
2. Stosować **agregacje**, **pivot** i **grupowanie** danych w PySpark
3. Łączyć dane za pomocą **join** (różne typy złączeń)
4. Używać **window functions** do rankingów i analiz okienkowych
5. Pisać zapytania **Spark SQL** i tworzyć **UDF** (User Defined Functions)

## 📋 Wymagania wstępne
- Podstawy Pythona (funkcje, listy, słowniki)
- Podstawy PySpark — tworzenie SparkSession, ładowanie danych, proste transformacje
- Znajomość SQL na poziomie podstawowym (SELECT, WHERE, GROUP BY)

---
> 💡 **Tip:** Uruchom każdą komórkę za pomocą `Shift+Enter` lub kliknij przycisk ▶️.  
> Jeśli utkniesz — rozwiń komórkę z podpowiedzią (kliknij) lub zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — Uruchom tę komórkę jako pierwszą!

# Instalacja PySpark
!pip install -q pyspark

# Importy
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, DateType
)

import matplotlib.pyplot as plt
import random
import datetime

# Konfiguracja wyświetlania
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Seed dla powtarzalności wyników
random.seed(42)

# Tworzenie SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("StreamPL-Analytics")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Ustawienie poziomu logowania na ERROR (mniej szumu w outputcie)
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ PySpark {spark.version} gotowy do pracy!")
print(f"🎬 StreamPL Analytics Engine uruchomiony 🚀")

## 📖 Wprowadzenie

W tym laboratorium będziemy pracować z **Apache Spark SQL** — jednym z najważniejszych modułów ekosystemu Spark. Spark SQL pozwala na przetwarzanie ustrukturyzowanych danych za pomocą zarówno API **DataFrame** (programistycznego), jak i czystego **SQL**.

Dlaczego Spark SQL jest tak potężny? Wyobraź sobie, że masz terabajty danych o oglądaniu filmów 🎬 — Spark SQL potrafi je przetworzyć na klastrze setek maszyn, a Ty piszesz kod tak samo jak dla małej tabelki w pandas!

### Kluczowe pojęcia

| Pojęcie | Opis | Przykład |
|---------|------|----------|
| **DataFrame** | Rozproszona kolekcja danych w formie tabeli (kolumny + wiersze) | `df = spark.read.csv("data.csv")` |
| **Transformacja** | Leniwa operacja tworząca nowy DataFrame (nie wykonuje się od razu!) | `df.filter(F.col("age") > 18)` |
| **Akcja** | Operacja wyzwalająca obliczenia i zwracająca wynik | `df.count()`, `df.show()` |
| **Window Function** | Funkcja analityczna działająca na "oknie" (podzbiorze) wierszy | `F.row_number().over(window)` |
| **UDF** | User Defined Function — własna funkcja Pythona w Spark | `F.udf(my_func, StringType())` |
| **Spark SQL** | Silnik SQL wbudowany w Spark — zapytania SQL na DataFrames | `spark.sql("SELECT * FROM ...")` |
| **Join** | Łączenie dwóch DataFrames po wspólnym kluczu | `df1.join(df2, "key")` |
| **Pivot** | Zamiana wartości w wierszach na kolumny (transpozycja) | `df.groupBy("x").pivot("y")` |

---

### 🎬 Scenariusz

Jesteś analitykiem danych w **StreamPL** — polskiej platformie streamingowej (jak Netflix, ale z polskimi serialami! 🇵🇱🎬). Twój szef poprosił Cię o przygotowanie raportu z danych o oglądaniu filmów i seriali za ostatni rok.

Masz do dyspozycji dane o ~200 000 sesji oglądania. Czas zabrać się do roboty! 💪

In [ ]:
# @title 📊 Generowanie danych StreamPL

# Lista tytułów filmów i seriali na platformie StreamPL 🎬
MOVIE_TITLES = [
    "Zimna Wojna", "Ida", "Boże Ciało", "Sala samobójców", "Kler",
    "Planeta Singli", "Pitbull", "Bogowie", "Drogówka", "Listy do M.",
    "Katyń", "Nad życie", "Dzień świra", "Miś", "Vabank",
    "Inception", "The Matrix", "Interstellar", "Parasite", "Joker",
    "Avengers: Endgame", "Dune", "Spider-Man: No Way Home",
    "Oppenheimer", "Barbie", "Top Gun: Maverick",
    "Wiedźmin: Sezon 1", "1670", "Rojst", "Belfer"
]

GENRES = ["Dramat", "Komedia", "Akcja", "Horror", "Sci-Fi", "Dokumentalny", "Animacja"]
DEVICES = ["TV", "Telefon", "Laptop", "Tablet"]
SUBSCRIPTION_TYPES = ["Basic", "Standard", "Premium"]
COUNTRIES = ["Polska", "Niemcy", "Czechy", "Słowacja", "Litwa"]

# Mapowanie tytuł -> gatunek (dla spójności danych)
TITLE_GENRE_MAP = {
    "Zimna Wojna": "Dramat", "Ida": "Dramat", "Boże Ciało": "Dramat",
    "Sala samobójców": "Dramat", "Kler": "Dramat",
    "Planeta Singli": "Komedia", "Pitbull": "Akcja", "Bogowie": "Dramat",
    "Drogówka": "Komedia", "Listy do M.": "Komedia",
    "Katyń": "Dramat", "Nad życie": "Dramat", "Dzień świra": "Komedia",
    "Miś": "Komedia", "Vabank": "Komedia",
    "Inception": "Sci-Fi", "The Matrix": "Sci-Fi", "Interstellar": "Sci-Fi",
    "Parasite": "Dramat", "Joker": "Dramat",
    "Avengers: Endgame": "Akcja", "Dune": "Sci-Fi",
    "Spider-Man: No Way Home": "Akcja",
    "Oppenheimer": "Dramat", "Barbie": "Komedia",
    "Top Gun: Maverick": "Akcja",
    "Wiedźmin: Sezon 1": "Akcja", "1670": "Komedia", "Rojst": "Horror",
    "Belfer": "Dramat"
}

# Generujemy ~200 000 wierszy danych
NUM_ROWS = 200_000
START_DATE = datetime.date(2025, 4, 13)
END_DATE = datetime.date(2026, 4, 12)
DATE_RANGE_DAYS = (END_DATE - START_DATE).days

random.seed(42)

rows = []
for _ in range(NUM_ROWS):
    title = random.choice(MOVIE_TITLES)
    genre = TITLE_GENRE_MAP[title]
    watch_date = START_DATE + datetime.timedelta(days=random.randint(0, DATE_RANGE_DAYS))
    # Ocena — ~10% wierszy ma None (brak oceny)
    rating = round(random.uniform(1.0, 5.0), 1) if random.random() > 0.1 else None

    rows.append((
        random.randint(1, 50_000),       # user_id
        title,                            # movie_title
        genre,                            # genre
        str(watch_date),                  # watch_date (jako string, przekonwertujemy)
        random.randint(5, 180),           # watch_duration_min
        rating,                           # rating (nullable)
        random.choice(DEVICES),           # device
        random.choice(SUBSCRIPTION_TYPES),# subscription_type
        random.choice(COUNTRIES),          # country
    ))

# Schema
schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("movie_title", StringType(), False),
    StructField("genre", StringType(), False),
    StructField("watch_date", StringType(), False),
    StructField("watch_duration_min", IntegerType(), False),
    StructField("rating", FloatType(), True),
    StructField("device", StringType(), False),
    StructField("subscription_type", StringType(), False),
    StructField("country", StringType(), False),
])

# Tworzenie DataFrame
df = spark.createDataFrame(rows, schema=schema)

# Konwersja watch_date ze stringa na DateType
df = df.withColumn("watch_date", F.col("watch_date").cast(DateType()))

# Cache — będziemy używać tego DataFrame wielokrotnie
df = df.cache()

print(f"🎬 StreamPL dataset: {df.count():,} wierszy")
print(f"📋 Kolumny: {df.columns}")
print(f"\n📊 Schema:")
df.printSchema()
print("\n🔍 Podgląd danych:")
df.show(5, truncate=False)

---

## 🎓 Przykład prowadzony

Zanim przejdziesz do ćwiczeń, przeanalizujmy wspólnie kilka operacji na naszym datasecie.

**Zadanie:** Znajdźmy 5 najczęściej oglądanych filmów na platformie i sprawdźmy ich średnią ocenę.

Użyjemy tutaj kombinacji `groupBy`, `agg`, `orderBy` i `join` — dokładnie tych technik, które będziesz stosować w ćwiczeniach.

In [ ]:
# @title 🎓 Przykład prowadzony — Uruchom i przeanalizuj

# Krok 1: Zliczamy ile razy każdy film był oglądany
watch_counts = (
    df.groupBy("movie_title")
    .agg(F.count("*").alias("total_views"))
    .orderBy(F.col("total_views").desc())
)
print("📊 Krok 1 — Liczba wyświetleń per film (top 5):")
watch_counts.show(5, truncate=False)

# Krok 2: Obliczamy średnią ocenę per film (pomijamy NULL)
avg_ratings = (
    df.groupBy("movie_title")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("rating").alias("rating_count")  # 👈 count pomija NULL!
    )
)
print("⭐ Krok 2 — Średnia ocena per film (top 5):")
avg_ratings.orderBy(F.col("avg_rating").desc()).show(5, truncate=False)

# Krok 3: Łączymy wyniki za pomocą join
top_movies = (
    watch_counts
    .join(avg_ratings, "movie_title")
    .orderBy(F.col("total_views").desc())
    .limit(5)
)

print("🏆 Krok 3 — Top 5 filmów: wyświetlenia + ocena:")
top_movies.show(truncate=False)

# Krok 4: To samo w Spark SQL!
df.createOrReplaceTempView("streaming")

top_movies_sql = spark.sql("""
    SELECT
        movie_title,
        COUNT(*) AS total_views,
        ROUND(AVG(rating), 2) AS avg_rating,
        COUNT(rating) AS rating_count
    FROM streaming
    GROUP BY movie_title
    ORDER BY total_views DESC
    LIMIT 5
""")

print("🔍 Krok 4 — Ten sam wynik w Spark SQL:")
top_movies_sql.show(truncate=False)

print("\n✅ Widać tę samą piątkę! DataFrame API i SQL dają identyczne wyniki 🎯")

---

## ✏️ Ćwiczenie 1: Eksploracja danych

**Trudność:** ⭐

Czas poznać nasz dataset! Wykonaj podstawową eksplorację danych:

1. Wyświetl schemat danych (`printSchema`)
2. Oblicz statystyki opisowe dla kolumn numerycznych (`describe`)
3. Policz liczbę wierszy per gatunek (`genre`) i posortuj malejąco

Zapisz wynik kroku 3 w zmiennej `genre_counts`.

**Oczekiwany output (krok 3):**
```
+------------+-----+
|genre       |count|
+------------+-----+
|Dramat      |XXXXX|
|Komedia     |XXXXX|
|Akcja       |XXXXX|
|...         |...  |
+------------+-----+
```

In [ ]:
# ✏️ Ćwiczenie 1: Eksploracja danych

# TODO 1: Wyświetl schemat danych
...

# TODO 2: Oblicz i wyświetl statystyki opisowe (describe)
...

# TODO 3: Policz liczbę wierszy per gatunek, posortuj malejąco
# Użyj: groupBy → count → orderBy (desc)
genre_counts = ...  # Zamień ... na swój kod

# Wyświetl wynik (nie zmieniaj tej linii)
genre_counts.show()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 1
print("""
Podpowiedź 1: Schema → df.printSchema()
Podpowiedź 2: Statystyki → df.describe().show()
Podpowiedź 3: Grupowanie i zliczanie:
   genre_counts = (
       df.groupBy("genre")
       .count()
       .orderBy(F.col("count").desc())
   )
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 1
try:
    assert genre_counts is not None and genre_counts is not ..., "❌ Zmienna 'genre_counts' nie została zdefiniowana"
    gc_rows = genre_counts.collect()
    assert len(gc_rows) == 7, f"❌ Oczekiwano 7 gatunków, otrzymano {len(gc_rows)}"
    # Sprawdź czy posortowane malejąco
    counts = [row["count"] for row in gc_rows]
    assert counts == sorted(counts, reverse=True), "❌ Wynik nie jest posortowany malejąco"
    # Sprawdź czy suma = NUM_ROWS
    assert sum(counts) == df.count(), "❌ Suma zliczeń nie zgadza się z liczbą wierszy"
    print("✅ Poprawnie! Świetna robota! 🎉")
    print(f"📊 Łącznie {sum(counts):,} wierszy w {len(gc_rows)} gatunkach")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 2: Filtrowanie i selekcja zaawansowana

**Trudność:** ⭐

Twój szef w StreamPL chce poznać zachowania użytkowników Premium z Polski 🇵🇱.

1. Dodaj nową kolumnę `watch_category` na podstawie `watch_duration_min`:
   - < 30 min → `"krótki"`
   - 30-60 min → `"średni"`
   - 60-120 min → `"długi"`
   - \> 120 min → `"maraton"`
   
   Użyj `F.when(...).when(...).otherwise(...)`

2. Przefiltruj dane: tylko użytkownicy **Premium** z **Polski** (`country == "Polska"`), którzy oglądali **dłużej niż 60 minut**.

Zapisz wynik (DataFrame z nową kolumną i po filtrze) w zmiennej `premium_pl`.

**Oczekiwany output:**
```
+-------+-------------------+------+----------+------------------+------+------+-----------------+-------+--------------+
|user_id|movie_title        |genre |watch_date|watch_duration_min|rating|device|subscription_type|country|watch_category|
+-------+-------------------+------+----------+------------------+------+------+-----------------+-------+--------------+
|...    |...                |...   |...       |>60               |...   |...   |Premium          |Polska |długi/maraton |
+-------+-------------------+------+----------+------------------+------+------+-----------------+-------+--------------+
```

In [ ]:
# ✏️ Ćwiczenie 2: Filtrowanie i selekcja zaawansowana

# TODO 1: Dodaj kolumnę watch_category za pomocą F.when()
# krótki: < 30 | średni: 30-60 | długi: 60-120 | maraton: > 120
df_with_category = df.withColumn(
    "watch_category",
    ...  # Zamień ... na wyrażenie F.when().when().when().otherwise()
)

# TODO 2: Filtruj — Premium + Polska + watch_duration_min > 60
premium_pl = df_with_category.filter(
    ...  # Zamień ... na warunki filtrowania
)

# Wyświetl wynik (nie zmieniaj)
print(f"Liczba wierszy po filtrze: {premium_pl.count():,}")
premium_pl.show(5, truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 2
print("""
Podpowiedź 1 — watch_category:
   F.when(F.col("watch_duration_min") < 30, "krótki")
    .when(F.col("watch_duration_min") < 60, "średni")
    .when(F.col("watch_duration_min") <= 120, "długi")
    .otherwise("maraton")

Podpowiedź 2 — Filtrowanie:
   Połącz warunki operatorem &
   (F.col("subscription_type") == "Premium") &
   (F.col("country") == "Polska") &
   (F.col("watch_duration_min") > 60)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 2
try:
    assert premium_pl is not None and premium_pl is not ..., "❌ Zmienna 'premium_pl' nie została zdefiniowana"
    assert "watch_category" in premium_pl.columns, "❌ Brakuje kolumny 'watch_category'"
    # Sprawdź filtrowanie
    subs = [row["subscription_type"] for row in premium_pl.select("subscription_type").distinct().collect()]
    assert subs == ["Premium"], f"❌ Powinny być tylko Premium, a mamy: {subs}"
    countries = [row["country"] for row in premium_pl.select("country").distinct().collect()]
    assert countries == ["Polska"], f"❌ Powinny być tylko Polska, a mamy: {countries}"
    min_dur = premium_pl.agg(F.min("watch_duration_min")).collect()[0][0]
    assert min_dur > 60, f"❌ Minimalny czas oglądania powinien być > 60, a jest {min_dur}"
    # Sprawdź kategorie
    cats = {row["watch_category"] for row in premium_pl.select("watch_category").distinct().collect()}
    assert cats.issubset({"długi", "maraton"}), f"❌ Po filtrze powinny być tylko 'długi' i 'maraton', a mamy: {cats}"
    print("✅ Poprawnie! Świetna robota! 🎉")
    print(f"📊 Znaleźliśmy {premium_pl.count():,} sesji Premium PL > 60 min")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 3: Agregacje i pivot

**Trudność:** ⭐⭐

Dział marketingu StreamPL chce wiedzieć, jak użytkownicy oglądają na różnych urządzeniach 📱💻📺.

**Część A:** Oblicz **średnią ocenę** (`rating`) per gatunek. Zapisz w `avg_rating_by_genre`.

**Część B:** Stwórz tabelę **pivot** — wiersze to `genre`, kolumny to `device`, wartości to **średni czas oglądania** (`watch_duration_min`). Zapisz w `pivot_genre_device`.

**Oczekiwany output (Część B):**
```
+------------+------+------+------+------+
|genre       |Laptop|Tablet|Telefon|TV   |
+------------+------+------+------+------+
|Dramat      |92.1  |91.5  |93.2  |92.8 |
|Komedia     |...   |...   |...   |...  |
+------------+------+------+------+------+
```

In [ ]:
# ✏️ Ćwiczenie 3: Agregacje i pivot

# Część A: Średnia ocena per gatunek
# TODO: groupBy genre, oblicz avg(rating), zaokrąglij do 2 miejsc, posortuj malejąco
avg_rating_by_genre = ...  # Zamień ... na swój kod

print("⭐ Średnia ocena per gatunek:")
avg_rating_by_genre.show()

# Część B: Pivot — średni czas oglądania per gatunek i urządzenie
# TODO: groupBy genre → pivot po device → avg watch_duration_min → zaokrąglij
pivot_genre_device = ...  # Zamień ... na swój kod

print("📊 Pivot — średni czas oglądania (min) per gatunek i urządzenie:")
pivot_genre_device.show(truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 3
print("""
Część A:
   avg_rating_by_genre = (
       df.groupBy("genre")
       .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
       .orderBy(F.col("avg_rating").desc())
   )

Część B — Pivot:
   pivot_genre_device = (
       df.groupBy("genre")
       .pivot("device")
       .agg(F.round(F.avg("watch_duration_min"), 1))
   )

Uwaga: pivot() zamienia unikalne wartości kolumny na nagłówki kolumn!
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 3
try:
    # Test Część A
    assert avg_rating_by_genre is not None and avg_rating_by_genre is not ..., "❌ Zmienna 'avg_rating_by_genre' nie została zdefiniowana"
    ar_rows = avg_rating_by_genre.collect()
    assert len(ar_rows) == 7, f"❌ Oczekiwano 7 gatunków, otrzymano {len(ar_rows)}"
    col_names = avg_rating_by_genre.columns
    assert "genre" in col_names, "❌ Brakuje kolumny 'genre'"
    assert "avg_rating" in col_names, "❌ Brakuje kolumny 'avg_rating' — użyj .alias('avg_rating')"
    # Sprawdź czy posortowane malejąco
    ratings = [row["avg_rating"] for row in ar_rows]
    assert ratings == sorted(ratings, reverse=True), "❌ Wynik nie jest posortowany malejąco po avg_rating"
    print("✅ Część A — Poprawnie!")

    # Test Część B
    assert pivot_genre_device is not None and pivot_genre_device is not ..., "❌ Zmienna 'pivot_genre_device' nie została zdefiniowana"
    pivot_cols = pivot_genre_device.columns
    for device in ["TV", "Telefon", "Laptop", "Tablet"]:
        assert device in pivot_cols, f"❌ Brakuje kolumny '{device}' w pivot"
    assert pivot_genre_device.count() == 7, f"❌ Oczekiwano 7 wierszy (gatunków) w pivot"
    print("✅ Część B — Poprawnie!")

    print("\n✅ Poprawnie! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 4: Joiny

**Trudność:** ⭐⭐

StreamPL dostało dane o budżetach filmów od działu zakupów 💰. Musisz połączyć te dane z naszym głównym datasetem.

**Krok 1:** Utwórz mały DataFrame `movies_info` na podstawie poniższych danych (uruchom komórkę poniżej).

**Krok 2:** Wykonaj `join` (left) `df` z `movies_info` po kolumnie `movie_title`.

**Krok 3:** Oblicz **sumaryczny czas oglądania** (suma `watch_duration_min`) per `genre`, ale tylko dla filmów, które mają `budget_mln` (nie NULL). Posortuj malejąco. Zapisz w `genre_watch_time`.

**Oczekiwany output (Krok 3):**
```
+------+--------------------+
|genre |total_watch_time_min|
+------+--------------------+
|Dramat|XXXXXXX             |
|...   |...                 |
+------+--------------------+
```

In [ ]:
# 📋 Dane o budżetach filmów (uruchom tę komórkę!)

movies_info_data = [
    ("Zimna Wojna", 10.5, 2018, "Paweł Pawlikowski"),
    ("Ida", 3.2, 2013, "Paweł Pawlikowski"),
    ("Boże Ciało", 5.1, 2019, "Jan Komasa"),
    ("Kler", 8.0, 2018, "Wojciech Smarzowski"),
    ("Planeta Singli", 6.0, 2016, "Sam Akina"),
    ("Pitbull", 12.0, 2005, "Patryk Vega"),
    ("Bogowie", 7.5, 2014, "Łukasz Palkowski"),
    ("Listy do M.", 5.5, 2011, "Mitja Okorn"),
    ("Inception", 160.0, 2010, "Christopher Nolan"),
    ("The Matrix", 63.0, 1999, "Wachowskis"),
    ("Interstellar", 165.0, 2014, "Christopher Nolan"),
    ("Parasite", 11.4, 2019, "Bong Joon-ho"),
    ("Joker", 55.0, 2019, "Todd Phillips"),
    ("Avengers: Endgame", 356.0, 2019, "Russo Brothers"),
    ("Dune", 165.0, 2021, "Denis Villeneuve"),
    ("Oppenheimer", 100.0, 2023, "Christopher Nolan"),
    ("Barbie", 145.0, 2023, "Greta Gerwig"),
    ("Top Gun: Maverick", 170.0, 2022, "Joseph Kosinski"),
]

movies_info_schema = StructType([
    StructField("movie_title", StringType(), False),
    StructField("budget_mln", FloatType(), True),
    StructField("release_year", IntegerType(), False),
    StructField("director", StringType(), False),
])

movies_info = spark.createDataFrame(movies_info_data, schema=movies_info_schema)

print("💰 Dane o budżetach filmów:")
movies_info.show(truncate=False)
print(f"📋 {movies_info.count()} filmów z danymi o budżecie")

In [ ]:
# ✏️ Ćwiczenie 4: Joiny

# TODO 1: Left join df z movies_info po kolumnie movie_title
df_joined = ...  # Zamień ... na swój kod

# TODO 2: Filtruj — tylko wiersze gdzie budget_mln nie jest NULL
df_with_budget = ...  # Zamień ... na swój kod

# TODO 3: Oblicz sumaryczny czas oglądania per gatunek, posortuj malejąco
genre_watch_time = ...  # Zamień ... na swój kod

# Wyświetl wynik (nie zmieniaj)
print("🎬 Sumaryczny czas oglądania per gatunek (filmy z budżetem):")
genre_watch_time.show(truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 4
print("""
Krok 1 — Left Join:
   df_joined = df.join(movies_info, "movie_title", "left")

Krok 2 — Filtrowanie NULL:
   df_with_budget = df_joined.filter(F.col("budget_mln").isNotNull())

Krok 3 — Agregacja:
   genre_watch_time = (
       df_with_budget
       .groupBy("genre")
       .agg(F.sum("watch_duration_min").alias("total_watch_time_min"))
       .orderBy(F.col("total_watch_time_min").desc())
   )
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 4
try:
    assert genre_watch_time is not None and genre_watch_time is not ..., "❌ Zmienna 'genre_watch_time' nie została zdefiniowana"
    gt_cols = genre_watch_time.columns
    assert "genre" in gt_cols, "❌ Brakuje kolumny 'genre'"
    assert "total_watch_time_min" in gt_cols, "❌ Brakuje kolumny 'total_watch_time_min'"
    gt_rows = genre_watch_time.collect()
    # Nie wszystkie gatunki muszą mieć budżet
    assert len(gt_rows) > 0, "❌ Wynik jest pusty"
    # Sprawdź sortowanie malejące
    times = [row["total_watch_time_min"] for row in gt_rows]
    assert times == sorted(times, reverse=True), "❌ Wynik nie jest posortowany malejąco"
    # Sprawdź że sumy są sensowne (> 0)
    assert all(t > 0 for t in times), "❌ Sumaryczny czas oglądania nie może być <= 0"
    print("✅ Poprawnie! Świetna robota! 🎉")
    print(f"🎬 Znaleziono {len(gt_rows)} gatunków z danymi budżetowymi")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 5: Window Functions

**Trudność:** ⭐⭐⭐

Window functions (funkcje okienkowe) to jedne z najpotężniejszych narzędzi w Spark SQL! Pozwalają wykonywać obliczenia na "oknie" wierszy powiązanym z bieżącym wierszem — np. ranking, średnia krocząca, numer wiersza w grupie.

**Zadanie:** Znajdź **top 3 filmy w każdym gatunku** pod względem średniej oceny (rating).

1. Oblicz średnią ocenę per film
2. Zdefiniuj okno (Window) partycjonowane po `genre`, posortowane malejąco po `avg_rating`
3. Użyj `F.row_number()` i `F.rank()` nad tym oknem
4. Przefiltruj do top 3 per gatunek (użyj `rank`)

Zapisz wynik w `top3_per_genre`.

**Oczekiwany output:**
```
+------+-------------------+----------+----------+----+
|genre |movie_title        |avg_rating|row_number|rank|
+------+-------------------+----------+----------+----+
|Akcja |Film X             |4.XX      |1         |1   |
|Akcja |Film Y             |4.XX      |2         |2   |
|Akcja |Film Z             |3.XX      |3         |3   |
|Dramat|...                |...       |1         |1   |
+------+-------------------+----------+----------+----+
```

In [ ]:
# ✏️ Ćwiczenie 5: Window Functions

# Krok 1: Oblicz średnią ocenę per film
# TODO: groupBy movie_title (i genre!), oblicz avg(rating)
movie_avg_rating = ...  # Zamień ... na swój kod

# Krok 2: Zdefiniuj okno
# TODO: Window.partitionBy("genre").orderBy(desc avg_rating)
window_spec = ...  # Zamień ... na swój kod

# Krok 3: Dodaj kolumny row_number i rank
# TODO: withColumn("row_number", F.row_number().over(window_spec))
#        withColumn("rank", F.rank().over(window_spec))
ranked = ...  # Zamień ... na swój kod

# Krok 4: Filtruj top 3 per gatunek
# TODO: filter rank <= 3
top3_per_genre = ...  # Zamień ... na swój kod

# Wyświetl wynik (nie zmieniaj)
print("🏆 Top 3 filmy per gatunek (wg średniej oceny):")
top3_per_genre.show(30, truncate=False)

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 5
print("""
Krok 1:
   movie_avg_rating = (
       df.groupBy("movie_title", "genre")
       .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
   )

Krok 2:
   window_spec = Window.partitionBy("genre").orderBy(F.col("avg_rating").desc())

Krok 3:
   ranked = (
       movie_avg_rating
       .withColumn("row_number", F.row_number().over(window_spec))
       .withColumn("rank", F.rank().over(window_spec))
   )

Krok 4:
   top3_per_genre = ranked.filter(F.col("rank") <= 3)

Różnica row_number vs rank:
  - row_number: zawsze kolejne 1,2,3... (nawet przy remisach)
  - rank: 1,1,3... (remisy dostają ten sam numer, następny jest pomijany)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 5
try:
    assert top3_per_genre is not None and top3_per_genre is not ..., "❌ Zmienna 'top3_per_genre' nie została zdefiniowana"
    t3_cols = top3_per_genre.columns
    for col_name in ["genre", "movie_title", "avg_rating", "rank"]:
        assert col_name in t3_cols, f"❌ Brakuje kolumny '{col_name}'"
    # Sprawdź że rank <= 3 wszędzie
    max_rank = top3_per_genre.agg(F.max("rank")).collect()[0][0]
    assert max_rank <= 3, f"❌ Maksymalny rank powinien być <= 3, a jest {max_rank}"
    # Sprawdź że mamy wyniki dla wszystkich gatunków
    n_genres = top3_per_genre.select("genre").distinct().count()
    assert n_genres == 7, f"❌ Oczekiwano 7 gatunków, otrzymano {n_genres}"
    print("✅ Poprawnie! Świetna robota! 🎉")
    print(f"🏆 Top 3 filmy dla {n_genres} gatunków — łącznie {top3_per_genre.count()} wierszy")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 6: Spark SQL

**Trudność:** ⭐⭐

Czas pisać czyste SQL w Spark! 📝 Twój szef jest oldschoolowy i lubi SQL bardziej niż DataFrame API (klasyczny manager 😄). Przygotuj dla niego kilka raportów.

**Krok 1:** Zarejestruj główny DataFrame jako widok tymczasowy `streaming` (jeśli jeszcze nie jest).

**Krok 2:** Napisz zapytanie SQL, które zwróci **liczbę aktywnych użytkowników (unikalnych user_id) per miesiąc**. Kolumny: `month`, `active_users`. Posortuj po `month`. Zapisz wynik w `monthly_active`.

**Krok 3:** Napisz zapytanie SQL, które znajdzie **"trending" gatunki** — te, które miały **powyżej 10 000 wyświetleń** i **średnią ocenę > 2.9**. Kolumny: `genre`, `view_count`, `avg_rating`. Posortuj po `view_count` malejąco. Zapisz w `trending_genres`.

**Oczekiwany output (Krok 2):**
```
+-------+------------+
|month  |active_users|
+-------+------------+
|2025-04|XXXXX       |
|2025-05|XXXXX       |
|...    |...         |
+-------+------------+
```

In [ ]:
# ✏️ Ćwiczenie 6: Spark SQL

# Krok 1: Zarejestruj widok tymczasowy
df.createOrReplaceTempView("streaming")

# Krok 2: Aktywni użytkownicy per miesiąc
# TODO: Napisz zapytanie SQL — DATE_FORMAT(watch_date, 'yyyy-MM') as month,
#        COUNT(DISTINCT user_id) as active_users
monthly_active = spark.sql("""
    ...
""")

print("📅 Aktywni użytkownicy per miesiąc:")
monthly_active.show(15)

# Krok 3: Trending gatunki
# TODO: SELECT genre, COUNT(*) view_count, ROUND(AVG(rating), 2) avg_rating
#        WHERE ... HAVING ...
trending_genres = spark.sql("""
    ...
""")

print("🔥 Trending gatunki:")
trending_genres.show()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 6
print("""
Krok 2 — Aktywni użytkownicy:
   SELECT
       DATE_FORMAT(watch_date, 'yyyy-MM') AS month,
       COUNT(DISTINCT user_id) AS active_users
   FROM streaming
   GROUP BY DATE_FORMAT(watch_date, 'yyyy-MM')
   ORDER BY month

Krok 3 — Trending gatunki:
   SELECT
       genre,
       COUNT(*) AS view_count,
       ROUND(AVG(rating), 2) AS avg_rating
   FROM streaming
   GROUP BY genre
   HAVING COUNT(*) > 10000 AND AVG(rating) > 2.9
   ORDER BY view_count DESC

Uwaga: HAVING filtruje po agregacji (jak WHERE, ale dla GROUP BY)
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 6
try:
    # Test Krok 2
    assert monthly_active is not None and monthly_active is not ..., "❌ Zmienna 'monthly_active' nie została zdefiniowana"
    ma_cols = monthly_active.columns
    assert "month" in ma_cols, "❌ Brakuje kolumny 'month'"
    assert "active_users" in ma_cols, "❌ Brakuje kolumny 'active_users'"
    ma_rows = monthly_active.collect()
    assert len(ma_rows) >= 12, f"❌ Oczekiwano >= 12 miesięcy, otrzymano {len(ma_rows)}"
    # Sprawdź sortowanie po miesiącu
    months = [row["month"] for row in ma_rows]
    assert months == sorted(months), "❌ Wynik nie jest posortowany po miesiącu"
    print("✅ Krok 2 — Poprawnie!")

    # Test Krok 3
    assert trending_genres is not None and trending_genres is not ..., "❌ Zmienna 'trending_genres' nie została zdefiniowana"
    tg_cols = trending_genres.columns
    assert "genre" in tg_cols, "❌ Brakuje kolumny 'genre'"
    assert "view_count" in tg_cols, "❌ Brakuje kolumny 'view_count'"
    assert "avg_rating" in tg_cols, "❌ Brakuje kolumny 'avg_rating'"
    tg_rows = trending_genres.collect()
    # Sprawdź HAVING
    for row in tg_rows:
        assert row["view_count"] > 10000, f"❌ {row['genre']} ma view_count={row['view_count']} (powinno > 10000)"
        assert row["avg_rating"] > 2.9, f"❌ {row['genre']} ma avg_rating={row['avg_rating']} (powinno > 2.9)"
    # Sprawdź sortowanie malejące
    views = [row["view_count"] for row in tg_rows]
    assert views == sorted(views, reverse=True), "❌ Wynik nie jest posortowany malejąco po view_count"
    print("✅ Krok 3 — Poprawnie!")

    print("\n✅ Poprawnie! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## ✏️ Ćwiczenie 7: UDF — własna funkcja

**Trudność:** ⭐⭐⭐

Czasami wbudowane funkcje Spark nie wystarczą i potrzebujemy **UDF** (User Defined Function) — naszej własnej funkcji Pythona, którą Spark aplikuje na każdy wiersz.

**Zadanie:**

1. Napisz funkcję `categorize_duration(minutes: int) -> str` która zwraca:
   - `"krótki"` — jeśli < 30 min
   - `"średni"` — jeśli 30-60 min
   - `"długi"` — jeśli 60-120 min
   - `"maraton"` — jeśli > 120 min

2. Zarejestruj ją jako UDF w Spark

3. Zastosuj UDF do DataFrame — dodaj kolumnę `duration_category`

4. Oblicz **rozkład kategorii** (ile % sesji w każdej kategorii). Zapisz w `duration_distribution`.

**Oczekiwany output:**
```
+-----------------+-----+----------+
|duration_category|count|percentage|
+-----------------+-----+----------+
|krótki           |XXXXX|XX.X      |
|średni           |XXXXX|XX.X      |
|długi            |XXXXX|XX.X      |
|maraton          |XXXXX|XX.X      |
+-----------------+-----+----------+
```

In [ ]:
# ✏️ Ćwiczenie 7: UDF — własna funkcja

# TODO 1: Napisz funkcję categorize_duration
def categorize_duration(minutes: int) -> str:
    """Kategoryzuje czas oglądania na: krótki, średni, długi, maraton."""
    ...  # Zamień ... na swój kod

# TODO 2: Zarejestruj jako UDF
# Użyj: F.udf(funkcja, StringType())
categorize_udf = ...  # Zamień ... na swój kod

# TODO 3: Zastosuj UDF — dodaj kolumnę duration_category
df_with_duration = ...  # Zamień ... na swój kod

# TODO 4: Oblicz rozkład kategorii
# groupBy duration_category → count → dodaj kolumnę percentage (count / total * 100)
total_rows = df.count()
duration_distribution = ...  # Zamień ... na swój kod

# Wyświetl wynik (nie zmieniaj)
print("📊 Rozkład kategorii czasu oglądania:")
duration_distribution.show()

In [ ]:
# @title 💡 Podpowiedź — Ćwiczenie 7
print("""
Krok 1 — Funkcja:
   def categorize_duration(minutes: int) -> str:
       if minutes < 30:
           return "krótki"
       elif minutes < 60:
           return "średni"
       elif minutes <= 120:
           return "długi"
       else:
           return "maraton"

Krok 2 — UDF:
   categorize_udf = F.udf(categorize_duration, StringType())

Krok 3 — Zastosowanie:
   df_with_duration = df.withColumn(
       "duration_category",
       categorize_udf(F.col("watch_duration_min"))
   )

Krok 4 — Rozkład:
   duration_distribution = (
       df_with_duration.groupBy("duration_category")
       .count()
       .withColumn("percentage", F.round(F.col("count") / total_rows * 100, 1))
       .orderBy("count")
   )

💡 UDF jest wolniejszy niż wbudowane F.when(), bo Spark nie może go zoptymalizować.
   W produkcji preferuj wbudowane funkcje — UDF to ostateczność.
""")

In [ ]:
# @title ✅ Test — Ćwiczenie 7
try:
    # Test funkcji Pythona
    assert categorize_duration(10) == "krótki", f"❌ categorize_duration(10) powinno być 'krótki', jest '{categorize_duration(10)}'"
    assert categorize_duration(45) == "średni", f"❌ categorize_duration(45) powinno być 'średni', jest '{categorize_duration(45)}'"
    assert categorize_duration(90) == "długi", f"❌ categorize_duration(90) powinno być 'długi', jest '{categorize_duration(90)}'"
    assert categorize_duration(150) == "maraton", f"❌ categorize_duration(150) powinno być 'maraton', jest '{categorize_duration(150)}'"
    print("✅ Funkcja categorize_duration — OK!")

    # Test DataFrame
    assert duration_distribution is not None and duration_distribution is not ..., "❌ Zmienna 'duration_distribution' nie została zdefiniowana"
    dd_cols = duration_distribution.columns
    assert "duration_category" in dd_cols, "❌ Brakuje kolumny 'duration_category'"
    assert "count" in dd_cols, "❌ Brakuje kolumny 'count'"
    assert "percentage" in dd_cols, "❌ Brakuje kolumny 'percentage'"
    dd_rows = duration_distribution.collect()
    categories = {row["duration_category"] for row in dd_rows}
    assert categories == {"krótki", "średni", "długi", "maraton"}, f"❌ Oczekiwano 4 kategorie, otrzymano: {categories}"
    # Sprawdź sumy
    total_count = sum(row["count"] for row in dd_rows)
    assert total_count == total_rows, f"❌ Suma count ({total_count}) != liczba wierszy ({total_rows})"
    total_pct = sum(row["percentage"] for row in dd_rows)
    assert abs(total_pct - 100.0) < 1.0, f"❌ Suma procentów powinna być ~100%, jest {total_pct}%"
    print("✅ Rozkład kategorii — OK!")

    print("\n✅ Poprawnie! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")
except Exception as e:
    print(f"❌ Coś poszło nie tak: {e}")

---

## 🏆 Bonus: Dashboard analityczny

**Trudność:** ⭐⭐⭐

Czas na wielki finał! 🎬 Połącz wszystkie techniki, których się nauczyłeś/aś, i stwórz mini **dashboard analityczny** dla StreamPL.

Twój dashboard powinien zawierać:

1. **SQL View:** Zarejestruj widok `streaming_enriched` z kolumnami: `month`, `genre`, `device`, `watch_duration_min`, `rating`, `subscription_type`

2. **Zapytanie SQL z window function:** Dla każdego miesiąca i gatunku oblicz łączną liczbę wyświetleń oraz ranking gatunków w danym miesiącu (top gatunek = rank 1)

3. **Wizualizacja matplotlib:** Stwórz wykres słupkowy (bar chart) pokazujący **łączny czas oglądania per gatunek** (w godzinach). Użyj danych z Spark → `.toPandas()` → `matplotlib`.

To ćwiczenie jest **otwarte** — nie ma jednego poprawnego rozwiązania. Baw się dobrze! 🎨

In [ ]:
# 🏆 Bonus: Dashboard analityczny

# --- Część 1: SQL View ---
# TODO: Zarejestruj widok streaming_enriched
# Dodaj kolumnę month = DATE_FORMAT(watch_date, 'yyyy-MM')
df_enriched = df.withColumn("month", F.date_format("watch_date", "yyyy-MM"))
df_enriched.createOrReplaceTempView("streaming_enriched")

# --- Część 2: SQL + Window Function ---
# TODO: Napisz zapytanie SQL które:
#   - Grupuje po month i genre
#   - Liczy wyświetlenia (COUNT(*))
#   - Rankuje gatunki w każdym miesiącu (RANK() OVER (PARTITION BY month ORDER BY count DESC))
monthly_genre_ranking = spark.sql("""
    ...
""")

print("📊 Ranking gatunków per miesiąc (top 3 per miesiąc):")
# Pokaż tylko top 3 z każdego miesiąca
monthly_genre_ranking.filter(F.col("genre_rank") <= 3).show(30, truncate=False)

# --- Część 3: Wizualizacja matplotlib ---
# TODO: Oblicz łączny czas oglądania per gatunek (w godzinach)
# Zamień na pandas → narysuj bar chart
watch_hours_pd = (
    df.groupBy("genre")
    .agg((F.sum("watch_duration_min") / 60).alias("total_hours"))
    .orderBy(F.col("total_hours").desc())
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#E50914", "#B81D24", "#F5F5F1", "#221F1F", "#564d4d", "#831010", "#c4302b"]
ax.bar(watch_hours_pd["genre"], watch_hours_pd["total_hours"], color=colors)
ax.set_xlabel("Gatunek", fontsize=12)
ax.set_ylabel("Łączny czas oglądania (godziny)", fontsize=12)
ax.set_title("🎬 StreamPL — Łączny czas oglądania per gatunek", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("\n🎉 Dashboard gotowy! Szef będzie zadowolony 📈")

---

## 📝 Podsumowanie

Gratulacje! 🎉 Ukończyłeś/aś Laboratorium 3 z kursu **Python w Zastosowaniach Big Data**.

### Czego się nauczyłeś/aś:

| Temat | Techniki |
|-------|----------|
| 📊 Eksploracja | `printSchema()`, `describe()`, `groupBy().count()` |
| 🔍 Filtrowanie | `F.when().otherwise()`, `filter()`, `withColumn()` |
| 📈 Agregacje | `agg()`, `avg()`, `sum()`, `count()`, `pivot()` |
| 🔗 Joiny | `join()` — left, inner, outer |
| 🪟 Window Functions | `Window.partitionBy()`, `row_number()`, `rank()` |
| 📝 Spark SQL | `createOrReplaceTempView()`, `spark.sql()`, `HAVING`, `DATE_FORMAT` |
| 🛠️ UDF | `F.udf()`, rejestracja własnych funkcji Pythona |
| 📊 Wizualizacja | Spark → `.toPandas()` → `matplotlib` |

### 📖 Materiały dodatkowe

- [Spark SQL Guide (oficjalna dokumentacja)](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)
- [Window Functions in Spark](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)
- [PySpark by Examples](https://sparkbyexamples.com/pyspark-tutorial/)

### 💡 Co dalej?

- **Laboratorium 4:** Spark Streaming i przetwarzanie danych w czasie rzeczywistym
- **Zadanie domowe:** Rozbuduj dashboard o dodatkowe metryki (retencja użytkowników, churn rate)

---

> 🎬 *"Dane są nową ropą naftową, ale tylko jeśli umiesz je rafinować."* — Clive Humby (parafrazując)
>
> Miłego kodowania! 🚀 — Zespół StreamPL Analytics